Maria Navarro  
February 11, 2024  
Healthcare Data & Analytics Case Study: Analytics of Patients and Consumers Survey  

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact

C:\Users\mfncn\AppData\Local\Temp\ipykernel_9648\4272756302.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
fall = pd.read_csv('mcbspuf21\\puf2021_1_fall.csv', index_col = None)
winter = pd.read_csv('mcbspuf21\\puf2021_2_winter.csv', index_col = None)
summer = pd.read_csv('mcbspuf21\\puf2021_3_summer.csv', index_col = None)

C:\Users\mfncn\AppData\Local\Temp\ipykernel_9648\507762521.py:1: DtypeWarning: Columns (43,46,49,50,51,52,53,73,76,77,81,82,83,85,90,92,99,100,105,106,107,112,113,118,119,123,124,125,126,128,129,135,136,137,138,139,140,142,143,151,155,156,161,163,172,176,183,184,185,186,191,192,193,194,195,197,198,199,200,208,209,210,211,214,217,218,219,220,221,222,223,224,225,227,229) have mixed types. Specify dtype option on import or set low_memory=False.
  fall = pd.read_csv('mcbspuf21\\puf2021_1_fall.csv', index_col = None)
C:\Users\mfncn\AppData\Local\Temp\ipykernel_9648\507762521.py:2: DtypeWarning: Columns (19,26,27,29,30,31,32,33,34,35,36,37,38,53,68,70,71,72,76,78,79,80,93) have mixed types. Specify dtype option on import or set low_memory=False.
  winter = pd.read_csv('mcbspuf21\\puf2021_2_winter.csv', index_col = None)


In [4]:
# Merging fall and winter with left join
fall_winter = pd.merge(fall, winter, on='PUF_ID', how='left')

In [5]:
# Merging all three datasets with left join
raw_data = pd.merge(fall_winter, summer, on='PUF_ID', how='left')

**Question 1: Racial disparity in ability to pay for care**  
In theory the responders in the MCBS are all insured by the Medicare. Most of them are eligible by age meaning they are seniors who have reached the retirement age 65 which is the eligibility criterion for Medicare. However, as we discussed earlier in class a small group have become eligible due to certain disabilities or other conditions. In any event while these people are all insured by Medicare program there are still elements of direct and indirect costs associated with utilization of healthcare services. We want to investigate if there is any racial disparity in terms of affordability of the out of pocket healthcare costs.  

In [6]:
# Filtering the data for those who are eligible for Medicare only because of age>65 and race
over_65 = raw_data['ADM_H_MEDSTA'] == 1
white_black = (raw_data['DEM_RACE'] == 1) | (raw_data['DEM_RACE'] ==2)
data = raw_data[over_65 & white_black]

In [7]:
# Note the following
data['ACC_HCDELAY'].unique()

array(['2', '1', 'D', 'R', nan, 2, 1], dtype=object)

In [8]:
# Handle incorrect formatting of ACC_HCDELAY column
data.loc[:,'ACC_HCDELAY'] = data['ACC_HCDELAY'].astype(str)

In [9]:
# Group by race and calculate value counts for each category
counts_by_race = data.groupby('DEM_RACE')['ACC_HCDELAY'].value_counts()

# Get total count for each gender group
total_counts_by_race = data['DEM_RACE'].value_counts()

In [10]:
# Calculate proportions for each category and convert it to a dataframe
proportions_by_race = counts_by_race / total_counts_by_race
proportions_by_race = proportions_by_race.reset_index(name='proportion')

In [11]:
proportions_by_race

,DEM_RACE,ACC_HCDELAY,proportion
0,1,2,0.957152
1,1,1,0.041252
2,1,nan,0.000859
3,1,D,0.000491
4,1,R,0.000246
5,2,2,0.938903
6,2,1,0.059850
7,2,nan,0.001247


In [12]:
# Use counts by race for Fisher's test
counts_by_race_filt = counts_by_race.reset_index(name='ACC_HCDELAY_Count')

# Building a 2x2 table with demographic race and acc_hcdelay
counts_by_race_filt = counts_by_race_filt[(counts_by_race_filt['ACC_HCDELAY']=='1') | (counts_by_race_filt['ACC_HCDELAY']=='2')]

q1_contingency_table = counts_by_race_filt.pivot_table(index='DEM_RACE', columns='ACC_HCDELAY', values='ACC_HCDELAY_Count')

In [13]:
q1_contingency_table

ACC_HCDELAY,1,2
DEM_RACE,,
1,336.0,7796.0
2,48.0,753.0


In [14]:
# Perform Fisher's exact test
'''
Running a statistical test exploring race differences in affordability of out of pocket healthcare costs
Null hypothesis (H0): Assumes that there is no association between race (1=white, 2=black) and delay in care for money reasons (1=yes, 2=no)
Alternative Hypothesis (H1): Assumes that there is a significant association between race (1=white, 2=black) and delay in care for money reasons (1=yes, 2=no)
P-value: The probability of obtaining results as extreme as the observed results, assuming the null hypothesis is true
'''
odds_ratio, p_value = fisher_exact(q1_contingency_table, alternative = 'two-sided')
alpha = 0.05

# Output the odds ratio and p-value based on the counts in the contingency table
print(f'Alpha: {alpha}')
print(f'P-value: {p_value}')

Alpha: 0.05
P-value: 0.0172581851028927


In [15]:
if p_value < alpha:
    print('The p-value is much smaller than alpha, which suggests that the observed data is unlikely under the assumption of the null hypothesis, leading to the rejection of the null hypothesis in favor of the alternative hypothesis. Therefore, there is a significant association between race and affordability of out of pocket healthcare costs.')
else:
    print('The p-value is greater than alpha, which suggests that the observed data is likely under the assumption of the null hypothesis, failing to reject the null hypothesis. Therefore, there is a not significant association between race and affordability of out of pocket healthcare costs.')

The p-value is much smaller than alpha, which suggests that the observed data is unlikely under the assumption of the null hypothesis, leading to the rejection of the null hypothesis in favor of the alternative hypothesis. Therefore, there is a significant association between race and affordability of out of pocket healthcare costs.


**Question 2: Gender differentials in healthcare utilization**  
Women on average utilize more health services. While during reproductive ages they do need more care we may think the gender differences should fade away older individuals once the child bearing related needs have passed. But again if they naturally have better health seeking behavior than men the trend of higher utilization should sustain as they age as well. Let us test this using our data. 

In [16]:
# Filter by age
data2 = raw_data[over_65].copy()

In [17]:
data2['ADM_H_PHYEVT'].unique()

array([4, 1, 0, 2, 3, 5], dtype=int64)

In [18]:
# Create a continuous variable for proxy ADM_H_PHYET

# Define a dictionary mappin goriginal values to the desired replacement values
replacement_dict = {
    1: 3,
    2: 8,
    3: 13,
    4: 18,
    5: 23
}

data2.loc[:,'hospital_visits'] = data2['ADM_H_PHYEVT'].replace(replacement_dict)

In [19]:
data2['hospital_visits'].describe()

count    10627.000000
mean         4.063141
std          6.178798
min          0.000000
25%          0.000000
50%          0.000000
75%          8.000000
max         23.000000
Name: hospital_visits, dtype: float64

In [20]:
data2['hospital_visits'].value_counts()

hospital_visits
0     5804
3     1871
8     1404
13     712
18     430
23     406
Name: count, dtype: int64

In [21]:
# Calculate the weighted average for male subjects
male_data= data2[data2['DEM_SEX']==1].copy()

# Calculate the length of male data or total number of male subjects
total_male = len(male_data)

# Calculate the ffrequency of each value in the hospital visits column for male subjects. It normalizes the counts to obtain the proportions instead of raw counts
male_visit_frequency = male_data['hospital_visits'].value_counts()
#male_visit_frequency = male_data['hospital_visits'].value_counts(normalize=True)

# Create new column where each value corresponds to the frequency of the corresponding value in the hospital_visits column
#male_data.loc[:,'visit_frequency'] = male_data['hospital_visits'].map(male_visit_frequency)

# Multiply each number of hospital visits by its corresponding visit frequency (weight), summing up the weighted values, and then dividing by the total number of male subjects
#weighted_avg_male = (male_data['hospital_visits'] * male_data['visit_frequency']).sum() / len(male_data)

In [31]:
# Corrected: Calculating the weighted average for male subjects
male_freq_sum = (male_visit_frequency.index.to_numpy()*male_visit_frequency.values).sum()
male_freq_sum
weighted_avg_male = male_freq_sum / total_male
weighted_avg_male

4.115767195767196

In [35]:
#male_data[['hospital_visits', 'visit_frequency']]
#male_visit_frequency
#(male_data['hospital_visits'] * male_data['visit_frequency']).sum()

,hospital_visits,visit_frequency
1,3,831
5,0,2571
7,0,2571
9,0,2571
16,8,617
...,...,...
12774,18,188
12775,13,328
12777,0,2571
12778,8,617


In [32]:
# Calculate the weighted average for female subjects
female_data= data2[data2['DEM_SEX']==2].copy()

# Calculate the length of female data
total_female = len(female_data)

#female_visit_frequency = female_data['hospital_visits'].value_counts(normalize=True)
female_visit_frequency = female_data['hospital_visits'].value_counts()
#female_data.loc[:,'visit_frequency'] = female_data['hospital_visits'].map(female_visit_frequency)
#weighted_avg_female = (female_data['hospital_visits'] * female_data['visit_frequency']).sum() / len(female_data)

# Corrected: Calculating the weighted average for male subjects
female_freq_sum = (female_visit_frequency.index.to_numpy()*female_visit_frequency.values).sum()
female_freq_sum
weighted_avg_female = female_freq_sum / total_female
weighted_avg_female

4.021009827177228

In [35]:
# Conducting a hypothesis test to test if the difference between these two averages is significant (two sample t-test)
from scipy import stats

t_statistic, p_value = stats.ttest_ind(a=[weighted_avg_male], b=[weighted_avg_female], equal_var = True)

# Determine significance
alpha = 0.05
if p_value < alpha:
    print("The difference between the two averages.")
else:
    print("The difference between the two averages is not significant.")


The difference between the two averages is not significant.


C:\Users\mfncn\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_stats_py.py:6988: RuntimeWarning: invalid value encountered in scalar divide
  svar = ((n1 - 1) * v1 + (n2 - 1) * v2) / df


**Question 3: A test of relationship between education and health**  
In the health econ textbook by Folland et al 7th Edition  in Chapter 7 it is suggested that education makes individuals healthier thru better life style and more informed life and health related decisions. Let us test this hypothesis here using the data in hand.  

In [29]:
df3 = raw_data.copy()


In [46]:
# This column has NAs
df3[df3['HLT_BMI_CAT'].isna()]

# Drop NAs
df3.dropna(subset=['HLT_BMI_CAT'], inplace=True)

In [48]:
# Create new obese column
df3['obese'] = np.where(df3['HLT_BMI_CAT']==3, 1, 0)

# Create new highly educated column
df3['highly_educated'] = np.where(df3['DEM_EDU']=='3', 1, 0)

In [49]:
df3['HLT_BMI_CAT'].value_counts()

HLT_BMI_CAT
2.0    4399
1.0    4094
3.0    4016
Name: count, dtype: int64

In [50]:
# Building a 2x2 table with obesity and education status
q3_contingency_table = pd.crosstab(df3['obese'], df3['highly_educated'])
q3_contingency_table

highly_educated,0,1
obese,,
0,3597,4896
1,2021,1995


In [51]:
# Perform Fisher's exact test
'''
Running a statistical test exploring differences between obesity rate and educational status
Null hypothesis (H0): Assumes that there is no association between education status (1=highly educated, 0=not highly educated) and obesity (1=obese, 0=not obese)
Alternative Hypothesis (H1): Assumes that there is a significant association between education status (1=highly educated, 0=not highly educated) and obesity (1=obese, 0=not obese)
P-value: The probability of obtaining results as extreme as the observed results, assuming the null hypothesis is true
'''
odds_ratio3, p_value3 = fisher_exact(q3_contingency_table, alternative = 'two-sided')
alpha = 0.05

# Output the odds ratio and p-value based on the counts in the contingency table
print(f'Alpha: {alpha}')
print(f'P-value: {p_value3}')

Alpha: 0.05
P-value: 7.320987972120796e-17


In [52]:
if p_value < alpha:
    print('The p-value is much smaller than alpha, which suggests that the observed data is unlikely under the assumption of the null hypothesis, leading to the rejection of the null hypothesis in favor of the alternative hypothesis. Therefore, there is a significant association between obesity and educational attainment.')
else:
    print('The p-value is greater than alpha, which suggests that the observed data is likely under the assumption of the null hypothesis, failing to reject the null hypothesis. Therefore, there is a not significant association between obesity and educational attainment.')

The p-value is much smaller than alpha, which suggests that the observed data is unlikely under the assumption of the null hypothesis, leading to the rejection of the null hypothesis in favor of the alternative hypothesis. Therefore, there is a significant association between obesity and educational attainment.
